In [3]:
import pandas as pd
import numpy as np
import matplotlib

# [FIX] Use non-interactive backend for stability
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# --- Configuration: File Paths ---
# Ensure these files contain 'leader_q_0'...'leader_q_6' and 'follower_q_0'...'follower_q_6'
PATH_SBSP = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/SBSP/high_var_cleaned.csv'
PATH_ASAC = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/ASAC/high_var.csv'
PATH_E2E  = '/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/data/high_var.csv'

# Settings
MAX_STEPS = 5000
TARGET_DT = 0.02

# Joint Columns
LEADER_J_COLS = [f'leader_q_{i}' for i in range(7)]
FOLLOWER_J_COLS = [f'follower_q_{i}' for i in range(7)]

def load_and_calc_joint_error(file_path, label):
    """
    Calculates the L2 Norm of the Joint Space Error (7-DoF).
    """
    try:
        # Load Data
        df = pd.read_csv(file_path)
        
        # --- Check for Joint Columns ---
        if not all(col in df.columns for col in LEADER_J_COLS + FOLLOWER_J_COLS):
            print(f"[{label}] Error: Joint columns (q_0...q_6) not found. Please check if cleaned file included them.")
            return None

        # --- Auto-Downsampling ---
        if 'time' in df.columns:
            df = df.sort_values('time').reset_index(drop=True)
            dt_original = df['time'].diff().median()
            
            if dt_original is not None and dt_original > 0:
                step_ratio = int(np.round(TARGET_DT / dt_original))
                if step_ratio > 1:
                    print(f"[{label}] Downsampling by factor {step_ratio}")
                    df = df.iloc[::step_ratio].reset_index(drop=True)

        # --- Truncate ---
        df = df.head(MAX_STEPS)

        # --- Joint Error Calculation ---
        # q_error = q_leader - q_follower
        q_leader = df[LEADER_J_COLS].to_numpy()
        q_follower = df[FOLLOWER_J_COLS].to_numpy()
        
        # Calculate Euclidean Norm of the 7-dim error vector
        # This aggregates vibration across all 7 joints into one metric
        joint_error_norm = np.linalg.norm(q_leader - q_follower, axis=1)
        
        return df.index.to_numpy(), joint_error_norm
        
    except FileNotFoundError:
        print(f"[{label}] Error: File not found.")
        return None
    except Exception as e:
        print(f"[{label}] Unexpected error: {e}")
        return None

def plot_joint_comparison(data_map):
    plt.figure(figsize=(12, 6))
    
    styles = {
        'SBSP': {'label': 'SBSP',   'c': '#ff7f0e', 'ls': '--', 'lw': 1.5, 'alpha': 0.9},
        'ASAC': {'label': 'ASAC',   'c': '#1f77b4', 'ls': '-',  'lw': 1.5, 'alpha': 0.9},
        'E2E':  {'label': 'E2E-RL', 'c': '#d62728', 'ls': '-',  'lw': 1.5, 'alpha': 0.9}
    }

    for key, val in data_map.items():
        if val is not None:
            steps, errors = val
            mean_err = np.mean(errors)
            s = styles[key]
            
            label_str = f"{s['label']} ($\mu$={mean_err:.3f} rad)"
            plt.plot(steps, errors, label=label_str, 
                     color=s['c'], linestyle=s['ls'], linewidth=s['lw'], alpha=s['alpha'])
            
            plt.axhline(y=mean_err, color=s['c'], linestyle='--', linewidth=1.0, alpha=0.5)

    # Formatting
    plt.title('Joint Space Tracking & Vibration Analysis (High Variance)', fontsize=14)
    plt.xlabel('Step', fontsize=12)
    plt.ylabel(r'Joint Error Norm $\|\mathbf{q}_{ideal} - \mathbf{q}_{actual}\|$ (rad)', fontsize=12)
    
    # Auto-scale Y to see vibration details (adjust if needed)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11, framealpha=0.95)
    plt.tight_layout()
    
    out_file = 'comparison_joint_vibration.png'
    plt.savefig(out_file, dpi=300)
    print(f"Joint comparison saved to: {out_file}")

if __name__ == "__main__":
    print("Loading Joint Data...")
    
    data = {
        'SBSP': load_and_calc_joint_error(PATH_SBSP, "SBSP"),
        'ASAC': load_and_calc_joint_error(PATH_ASAC, "ASAC"),
        'E2E':  load_and_calc_joint_error(PATH_E2E,  "E2E-RL")
    }
    
    if any(v is not None for v in data.values()):
        plot_joint_comparison(data)
    else:
        print("No valid data found. Note: Ensure your CSV files contain 'leader_q_0' etc.")

Loading Joint Data...
[E2E-RL] Downsampling by factor 4
Joint comparison saved to: comparison_joint_vibration.png
